# LiquiLens, Seiche and Undertow research desk

Keep `research_client.py` beside this notebook. Requires Python 3.10+; the capture client uses only standard libraries. This notebook reads public source-cited research tables. Your environment may require network approval.

Each product answers a different question. Inspect dates, units, availability, rights and sources before interpreting a result. Retrieval success is not evidence validation or financial authority.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
from research_client import collect, save_capture

# Set True for operator testing, demonstrations or CI rather than external research.
SYNTHETIC = False


## Start with three useful tasks

Review USD funding, covered ESAF bank disclosures and market-liquidity observations. The bank filter is a name/ID substring, not universal issuer coverage. Missing or restricted numbers remain null.


In [ ]:
tasks = {
    "funding": ("money_markets", "USD"),
    "bank": ("bank_risk", "ESAF"),
    "liquidity": ("market_liquidity", ""),
}
captures = {
    task: collect(dataset, entity=entity, synthetic=SYNTHETIC)
    for task, (dataset, entity) in tasks.items()
}
for task, capture in captures.items():
    print(task, capture["returned_rows"], "rows; captured", capture["captured_at"])


## Inspect observations and sources

`as_of`, `published_at` and `knowledge_time` are different clocks. `captured_at` only records this retrieval. History reflects currently published observations; it is not an as-published vintage archive. Keep page-level source hashes and diagnostics with the rows.


In [ ]:
columns = ("entity_id", "metric", "value", "unit", "as_of", "published_at",
           "knowledge_time", "availability", "source_url", "rights_status")
for task, capture in captures.items():
    print("\n", task)
    for row in capture["results"]:
        print(json.dumps({name: row.get(name) for name in columns}))
    print("Source metadata:", json.dumps(capture["pages"], indent=2))


## Save a review you can reproduce

This writes local files in a new dated directory. Each dataset gets original JSON, spreadsheet-safe CSV and checksum receipts. No files are overwritten. CSV blanks retain missing values; original strings remain in JSON. Refreshing later is a new capture, not proof that source observations advanced.


In [ ]:
capture_dir = Path("captures") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
for task, capture in captures.items():
    receipt = save_capture(capture, capture_dir / task)
    print(task, json.dumps(receipt))
print("Saved:", capture_dir.resolve())


## Continue in your own tools

- Import `rows.csv` into Excel, Power BI, R or DuckDB. Retain the original JSON and receipt.
- Optional pandas: `pd.DataFrame(captures["funding"]["results"])`; do not fill nulls with zero.
- Use the hosted OpenBB research desk at `https://api.seiche.info/openbb` or MCP at `https://api.seiche.info/openbb/mcp`.
- An entitled Bloomberg BQuant administrator can evaluate approved own-data import. This notebook has not been validated in BQuant or accepted into Bloomberg App Portal.
- See `setup.md`, `agent-instructions.md` and `vendor-readiness.md` in the kit. For updates: https://beepboop2025.github.io/financial-evidence-skills/tools/
